In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    RocCurveDisplay,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import joblib

pd.set_option("display.max_columns", None)

DATA_PATH = "mental_health_workplace_survey.csv"

In [ ]:
# Load dataset

df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
print("\nColumns:\n", df.columns.tolist())
print("\nData types:\n", df.dtypes)

# Quick look at target-related columns
print("\nTarget column unique values (BurnoutRisk):", df["BurnoutRisk"].unique())
if "BurnoutLevel" in df.columns:
    print("BurnoutLevel sample values:")
    print(df["BurnoutLevel"].describe())
else:
    print("BurnoutLevel not in df (dropped to avoid leakage).")

df.head()

In [ ]:
# Exploratory Data Analysis (EDA)

# 1. Missing values overview
missing_counts = df.isnull().sum().sort_values(ascending=False)
print("Missing values per column (top 20):")
print(missing_counts.head(20))

# 2. Basic statistics for numeric columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
print("\nNumeric columns:", numeric_cols.tolist())

df[numeric_cols].describe().T


In [ ]:
# 3. Target distribution and relationship with key features

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

sns.countplot(x="BurnoutRisk", data=df, ax=axes[0])
axes[0].set_title("BurnoutRisk distribution")

if "BurnoutLevel" in df.columns:
    sns.histplot(df["BurnoutLevel"], kde=True, ax=axes[1])
    axes[1].set_title("BurnoutLevel distribution")
    sns.boxplot(x="BurnoutRisk", y="BurnoutLevel", data=df, ax=axes[2])
    axes[2].set_title("BurnoutLevel vs BurnoutRisk")
else:
    axes[1].text(0.5, 0.5, "BurnoutLevel dropped (avoid leakage)", ha="center", va="center")
    axes[1].set_title("BurnoutLevel (excluded from model)")
    axes[2].text(0.5, 0.5, "N/A", ha="center", va="center")
    axes[2].set_title("BurnoutLevel vs BurnoutRisk")

plt.tight_layout()
plt.show()

# 4. Correlation matrix for main numeric features
plt.figure(figsize=(12, 10))
correlation = df[numeric_cols].corr()
sns.heatmap(correlation, cmap="coolwarm", center=0)
plt.title("Correlation heatmap (numeric features)")
plt.show()


In [ ]:
# 5. Example group-level insights

# Burnout risk by department
burnout_by_dept = df.groupby("Department")["BurnoutRisk"].mean().sort_values(ascending=False)
print("Burnout risk rate by Department (mean of BurnoutRisk):")
print(burnout_by_dept)

# Burnout risk by remote work status
burnout_by_remote = df.groupby("RemoteWork")["BurnoutRisk"].mean().sort_values(ascending=False)
print("\nBurnout risk rate by RemoteWork:")
print(burnout_by_remote)

# Burnout risk by having mental health support
burnout_by_support = df.groupby("HasMentalHealthSupport")["BurnoutRisk"].mean().sort_values(ascending=False)
print("\nBurnout risk rate by HasMentalHealthSupport:")
print(burnout_by_support)

In [ ]:
# Data preprocessing: define features and target

# Use BurnoutRisk as the primary target for classification
TARGET_COL = "BurnoutRisk"

# Make a working copy
data = df.copy()

# Ensure target is numeric (0/1)
data[TARGET_COL] = data[TARGET_COL].astype(int)

# Drop target and BurnoutLevel so the model never sees it (fixes data leakage)
cols_to_drop = [TARGET_COL] + [c for c in ["BurnoutLevel"] if c in data.columns]
X = data.drop(columns=cols_to_drop)
y = data[TARGET_COL]

# EmployeeID: keep in X for output display but do not use as a model feature
numeric_features = [c for c in X.select_dtypes(include=[np.number]).columns.tolist() if c != "EmployeeID"]
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric features (used for model; BurnoutLevel dropped to avoid leakage):", numeric_features)
print("Categorical features (used for model):", categorical_features)

# Define preprocessing for numeric and categorical data
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

# Train-test split (with stratification to preserve BurnoutRisk distribution)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_train.shape, X_test.shape

In [ ]:
# Baseline models: compare multiple algorithms

models = {
    "log_reg": LogisticRegression(max_iter=500, class_weight="balanced", n_jobs=-1),
    "rf": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample",
    ),
    "gb": GradientBoostingClassifier(random_state=42),
}

results = {}

for name, clf in models.items():
    pipe = Pipeline(steps=[("preprocessor", preprocessor), ("model", clf)])
    # Use cross-validation on the training set for a robust estimate
    cv_accuracy = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy")
    cv_precision = cross_val_score(pipe, X_train, y_train, cv=5, scoring="precision")
    cv_recall = cross_val_score(pipe, X_train, y_train, cv=5, scoring="recall")
    cv_f1 = cross_val_score(pipe, X_train, y_train, cv=5, scoring="f1")

    results[name] = {
        "accuracy_mean": cv_accuracy.mean(),
        "precision_mean": cv_precision.mean(),
        "recall_mean": cv_recall.mean(),
        "f1_mean": cv_f1.mean(),
    }

results_df = pd.DataFrame(results).T.sort_values(by="f1_mean", ascending=False)
print("Cross-validated model performance on training data:")
results_df

In [ ]:
# Train the best baseline model on the full training set

# You can change this to "log_reg" or "gb" if results_df suggests a better choice
best_model_name = "rf"

best_clf = models[best_model_name]

burnout_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", best_clf),
])

burnout_pipeline.fit(X_train, y_train)

# Predictions on test set
y_pred = burnout_pipeline.predict(X_test)
y_proba = burnout_pipeline.predict_proba(X_test)[:, 1] if hasattr(burnout_pipeline["model"], "predict_proba") else None

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else np.nan

print(f"Test Accuracy: {acc:.4f}")
print(f"Test Precision: {prec:.4f}")
print(f"Test Recall: {rec:.4f}")
print(f"Test F1-score: {f1:.4f}")
print(f"Test ROC-AUC: {roc_auc:.4f}")

print("\nClassification report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(confusion_matrix=cm).plot(cmap="Blues")
plt.title("Confusion Matrix - BurnoutRisk")
plt.show()

if y_proba is not None:
    RocCurveDisplay.from_predictions(y_test, y_proba)
    plt.title("ROC Curve - BurnoutRisk")
    plt.show()

In [ ]:
# Hyperparameter tuning for the chosen model (Random Forest by default)

from sklearn.model_selection import RandomizedSearchCV

rf_params = {
    "model__n_estimators": [200, 300, 400, 500],
    "model__max_depth": [None, 5, 10, 15, 20],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4],
    "model__max_features": ["sqrt", "log2", None],
}

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        random_state=42,
        n_jobs=-1,
        class_weight="balanced_subsample",
    )),
])

random_search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=rf_params,
    n_iter=20,
    scoring="f1",
    n_jobs=-1,
    cv=5,
    verbose=1,
    random_state=42,
)

random_search.fit(X_train, y_train)

print("Best parameters:")
print(random_search.best_params_)

print("\nBest cross-validated F1:", random_search.best_score_)

best_burnout_pipeline = random_search.best_estimator_

# Evaluate tuned model on the held-out test set
y_pred_tuned = best_burnout_pipeline.predict(X_test)
y_proba_tuned = best_burnout_pipeline.predict_proba(X_test)[:, 1]

acc_tuned = accuracy_score(y_test, y_pred_tuned)
prec_tuned = precision_score(y_test, y_pred_tuned)
rec_tuned = recall_score(y_test, y_pred_tuned)
f1_tuned = f1_score(y_test, y_pred_tuned)
roc_auc_tuned = roc_auc_score(y_test, y_proba_tuned)

print("\nTUNED MODEL PERFORMANCE ON TEST SET")
print(f"Accuracy: {acc_tuned:.4f}")
print(f"Precision: {prec_tuned:.4f}")
print(f"Recall: {rec_tuned:.4f}")
print(f"F1-score: {f1_tuned:.4f}")
print(f"ROC-AUC: {roc_auc_tuned:.4f}")


In [ ]:
# Optional: Inspect feature importances for interpretability

rf_model = best_burnout_pipeline["model"]

# Get feature names after preprocessing
cat_encoder = best_burnout_pipeline["preprocessor"].named_transformers_["cat"]["onehot"]
cat_feature_names = cat_encoder.get_feature_names_out(best_burnout_pipeline["preprocessor"].transformers_[1][2])

feature_names = np.concatenate([numeric_features, cat_feature_names])

importances = rf_model.feature_importances_
indices = np.argsort(importances)[::-1]

# Show top 20 most important features
n_top = 20
print("Top feature importances:")
for idx in indices[:n_top]:
    print(f"{feature_names[idx]}: {importances[idx]:.4f}")

plt.figure(figsize=(10, 6))
plt.title("Top Feature Importances - Burnout Risk Model")
plt.bar(range(n_top), importances[indices[:n_top]], align="center")
plt.xticks(range(n_top), feature_names[indices[:n_top]], rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================
# FIX: Threshold tuning (model predicts only class 0 at 0.5)
# Lower the decision threshold to get non-zero recall for burnout risk (class 1)
# =====================================================

if y_proba is not None:
    best_f1 = 0
    best_threshold = 0.5

    for thresh in np.arange(0.20, 0.55, 0.02):
        y_pred_th = (y_proba >= thresh).astype(int)
        f1_th = f1_score(y_test, y_pred_th, zero_division=0)
        if f1_th >= best_f1:
            best_f1 = f1_th
            best_threshold = thresh

    # Apply best threshold and report metrics
    y_pred_fixed = (y_proba >= best_threshold).astype(int)

    acc_f = accuracy_score(y_test, y_pred_fixed)
    prec_f = precision_score(y_test, y_pred_fixed, zero_division=0)
    rec_f = recall_score(y_test, y_pred_fixed, zero_division=0)
    f1_f = f1_score(y_test, y_pred_fixed, zero_division=0)

    print(f"Best threshold (max F1): {best_threshold:.2f}")
    print(f"\nMetrics at threshold {best_threshold:.2f}:")
    print(f"  Accuracy:  {acc_f:.4f}")
    print(f"  Precision: {prec_f:.4f}")
    print(f"  Recall:    {rec_f:.4f}")
    print(f"  F1-score:  {f1_f:.4f}")
    print(f"  ROC-AUC:   {roc_auc:.4f}")
    print("\nClassification report (with tuned threshold):")
    print(classification_report(y_test, y_pred_fixed))

    # For deployment: use predict_proba(X)[:, 1] >= best_threshold instead of predict(X)
    print(f"\n>>> Use threshold {best_threshold:.2f} for predictions: risk = (probability >= {best_threshold:.2f})")
else:
    print("No probabilities available; threshold tuning skipped.")

In [ ]:
# Save the final tuned pipeline for deployment/use in the application

MODEL_PATH = "burnout_risk_pipeline.joblib"

joblib.dump(best_burnout_pipeline, MODEL_PATH)

print(f"Saved tuned burnout detection pipeline to {MODEL_PATH}")

# Example: new data must have same columns as training (no BurnoutLevel - avoids leakage)
# new_employee = pd.DataFrame([
#     {
#         "EmployeeID": 9999,
#         "Age": 35,
#         "Gender": "Female",
#         "Country": "USA",
#         "JobRole": "Software Engineer",
#         "Department": "Engineering",
#         "YearsAtCompany": 3,
#         "WorkHoursPerWeek": 45,
#         "RemoteWork": "Hybrid",
#         "JobSatisfaction": 7.0,
#         "StressLevel": 6.0,
#         "ProductivityScore": 7.5,
#         "SleepHours": 7.0,
#         "PhysicalActivityHrs": 3.0,
#         "CommuteTime": 30,
#         "HasMentalHealthSupport": "Yes",
#         "ManagerSupportScore": 8.0,
#         "HasTherapyAccess": "Yes",
#         "MentalHealthDaysOff": 2,
#         "SalaryRange": "80K-100K",
#         "WorkLifeBalanceScore": 8.0,
#         "TeamSize": 10,
#         "CareerGrowthScore": 7.5,
#     }
# ])
# burnout_risk_proba = best_burnout_pipeline.predict_proba(new_employee)[:, 1]
# print("Predicted burnout risk probability:", burnout_risk_proba[0])

In [ ]:
# Post-processing: convert model outputs into score, category, and recommendations

def compute_burnout_outputs(pipeline, X_input: pd.DataFrame) -> pd.DataFrame:
    """Given features X_input, return burnout risk score, alert category, and recommendations."""
    if not hasattr(pipeline, "predict_proba"):
        raise ValueError("Pipeline must support predict_proba for risk scoring.")

    proba = pipeline.predict_proba(X_input)[:, 1]
    score_pct = (proba * 100).round(1)

    # Define thresholds for alert categories
    def categorize(p: float) -> str:
        if p < 0.33:
            return "Low"
        elif p < 0.66:
            return "Medium"
        else:
            return "High"

    categories = [categorize(p) for p in proba]

    def make_recommendation(cat: str) -> str:
        if cat == "High":
            return (
                "High burnout risk: redistribute workload, schedule immediate rest days, "
                "and provide access to mental health support (counseling, EAP)."
            )
        if cat == "Medium":
            return (
                "Medium burnout risk: review workload and deadlines, encourage regular breaks "
                "and time off, and monitor stress levels with frequent check-ins."
            )
        return (
            "Low burnout risk: maintain current workload balance, reinforce healthy habits "
            "(sleep, exercise), and continue periodic well-being check-ins."
        )

    recs = [make_recommendation(c) for c in categories]

    # Try to keep EmployeeID for easier mapping if present
    out = pd.DataFrame({
        "burnout_risk_score": score_pct,
        "alert_category": categories,
        "recommendations": recs,
    }, index=X_input.index)

    if "EmployeeID" in X_input.columns:
        out.insert(0, "EmployeeID", X_input["EmployeeID"].values)

    return out

# Example: generate outputs for a sample of the test set
# (Use best_burnout_pipeline if tuning cell was run, otherwise fallback to burnout_pipeline.)
final_pipeline = "best_burnout_pipeline" in globals() and best_burnout_pipeline or burnout_pipeline

burnout_outputs_sample = compute_burnout_outputs(final_pipeline, X_test.copy().head(10))

burnout_outputs_sample